In [1]:
from mlxtend.frequent_patterns import fpgrowth, association_rules
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [73]:
#sheet_name=None to read all sheets
dfs = pd.read_excel('/home/hady/Studies/Uni/Data Mining/Assignments/Assignment 1/Data/online_retail_II.xlsx', sheet_name=None)

In [74]:
dfs.keys() #two sheets

dict_keys(['Year 2009-2010', 'Year 2010-2011'])

In [75]:
#join 2 sheets into one dataframe

df = pd.concat(dfs.values(), ignore_index=True)

In [76]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [77]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


In [78]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,1.067371e+06,1067371,1.067371e+06,824364.000000
mean,9.938898e+00,2011-01-02 21:13:55.394029,4.649388e+00,15324.638504
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.727058e+02,NaN,1.235531e+02,1697.464450


# Preprocessing

We will drop rows where Customer ID is null since it won't help us in any way, We will also drop rows in Price and Quantitiy where the value is -ve since it might indicate cancellations or returns or not valid values in general. We will drop InvoiceDate and Country too since they are Irrelevant to what we are doing.

In [85]:
#drop cancellations and returns

df = df[df['Quantity'] > 0]
df = df[df['Price'] > 0]
df = df.dropna(subset=['Customer ID']) #drop rows with missing CustomerID

In [86]:
df.info()

<class 'pandas.DataFrame'>
Index: 805549 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      805549 non-null  object        
 1   StockCode    805549 non-null  object        
 2   Description  805549 non-null  object        
 3   Quantity     805549 non-null  int64         
 4   InvoiceDate  805549 non-null  datetime64[us]
 5   Price        805549 non-null  float64       
 6   Customer ID  805549 non-null  float64       
 7   Country      805549 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 55.3+ MB


In [87]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,805549.000000,805549,805549.000000,805549.000000
mean,13.290522,2011-01-02 10:24:44.106814,3.206561,15331.954970
min,1.000000,2009-12-01 07:45:00,0.001000,12346.000000
25%,2.000000,2010-07-07 12:08:00,1.250000,13982.000000
50%,5.000000,2010-12-03 15:10:00,1.950000,15271.000000
75%,12.000000,2011-07-28 13:05:00,3.750000,16805.000000
max,80995.000000,2011-12-09 12:50:00,10953.500000,18287.000000
std,143.634088,NaN,29.199173,1696.737039


There is still some huge outliers in quantity and price, for an e-commerce recommender these outliers will mess up association rules because they're not normal shopping behavior. We will cap them by IQR

In [88]:
q = df['Quantity'].quantile(0.99) #we notice the ((% of the quantity is 128 which is acceptable, so we will use it to deal with them

In [89]:
#Remove quantity outliers

df = df[df['Quantity'] <= q]

In [90]:
df.info()

<class 'pandas.DataFrame'>
Index: 797565 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      797565 non-null  object        
 1   StockCode    797565 non-null  object        
 2   Description  797565 non-null  object        
 3   Quantity     797565 non-null  int64         
 4   InvoiceDate  797565 non-null  datetime64[us]
 5   Price        797565 non-null  float64       
 6   Customer ID  797565 non-null  float64       
 7   Country      797565 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 54.8+ MB


In [91]:
df = df.drop(['InvoiceDate', 'Country'], axis=1)

In [92]:
df.head()

,Invoice,StockCode,Description,Quantity,Price,Customer ID
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,6.95,13085.0
1,489434,79323P,PINK CHERRY LIGHTS,12,6.75,13085.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,6.75,13085.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2.10,13085.0
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,1.25,13085.0


In [93]:
#ensure right data types

df['StockCode'] = df['StockCode'].astype(str)
df['Customer ID'] = df['Customer ID'].astype(int)
df['Description'] = df['Description'].astype(str)
df['Invoice'] = df['Invoice'].astype(int)

In [94]:
df.dtypes

Invoice          int64
StockCode          str
Description        str
Quantity         int64
Price          float64
Customer ID      int64
dtype: object

In [95]:
#dropping duplicate rows

df = df.drop_duplicates()

In [96]:
df.info()

<class 'pandas.DataFrame'>
Index: 771542 entries, 0 to 1067370
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      771542 non-null  int64  
 1   StockCode    771542 non-null  str    
 2   Description  771542 non-null  str    
 3   Quantity     771542 non-null  int64  
 4   Price        771542 non-null  float64
 5   Customer ID  771542 non-null  int64  
dtypes: float64(1), int64(3), str(2)
memory usage: 41.2 MB


In [97]:
#after finishing preprocessing, divide interactions and metadate for when needed later in the system

#save interaction file
interaction_df = df[['Customer ID', 'Invoice', 'StockCode', 'Quantity']]
interaction_df.to_csv('/home/hady/Studies/Uni/Data Mining/Assignments/Assignment 1/Data/interactions.csv', index=False)

#save metadata file
metadata_df = df[['StockCode', 'Description', 'Price']].drop_duplicates(subset='StockCode').reset_index(drop=True)
metadata_df.to_csv('/home/hady/Studies/Uni/Data Mining/Assignments/Assignment 1/Data/metadata.csv', index=False)

print(f"Interactions: {len(interaction_df)} rows")
print(f"Metadata: {len(metadata_df)} unique products")

Interactions: 771542 rows
Metadata: 4622 unique products


# Getting Rules

## Association Rules

1. Support — how often do these items appear together across ALL transactions
    - Low support = rare combination, High support = very common combination

2. Confidence — given that the customer bought item A, how likely are they to also buy item B
    - Example: conf = 0.7 means 70% of people who bought A also bought B, It's basically a conditional probability

3. Lift — is this association actually meaningful or just a coincidence?
    - Lift = 1 → pure coincidence, the items are independent
    - Lift > 1 → they actually go together more than random chance
    - Lift < 1 → buying A actually makes B less likely
    - We always want lift > 1, otherwise the rule is useless

## Approach

- We will use Invoice and StockCode for Association Rules as they represent a cart or a basket of products
- We will use StockCode and Descripition for TF-IDF to mine for words in descripition per product

In [2]:
interaction_df = pd.read_csv('/home/hady/Studies/Uni/Data Mining/Assignments/Assignment 1/Data/interactions.csv')
metadata_df = pd.read_csv('/home/hady/Studies/Uni/Data Mining/Assignments/Assignment 1/Data/metadata.csv')

In [3]:
basket = interaction_df.groupby(['Invoice', 'StockCode'])['Quantity'].sum().unstack().fillna(0)

#groupby → groups by invoice and product
#sum() → adds up quantities
#unstack() → pivots StockCode into columns
#fillna(0) → fills missing combinations with 0

basket = basket.map(lambda x: 1 if x > 0 else 0).astype(bool)

#Converts all numbers into just 0 or 1 (bought or not bought), because association rules don't care about quantity,
#just whether the item was present, converted to bool as fpgrowth algorithm works better with it

Number of rules: 244
           antecedents          consequents  antecedent support  \
0   frozenset({21755})   frozenset({21754})            0.041012   
1   frozenset({20971})   frozenset({20972})            0.026651   
2  frozenset({85014A})  frozenset({85014B})            0.017565   
3  frozenset({85014B})  frozenset({85014A})            0.021790   
4   frozenset({21929})  frozenset({85099B})            0.033169   

   consequent support   support  confidence       lift  representativity  \
0            0.050568  0.021707    0.529293  10.467050               1.0   
1            0.032892  0.015493    0.581347  17.674222               1.0   
2            0.021790  0.010937    0.622642  28.574431               1.0   
3            0.017565  0.010937    0.501901  28.574431               1.0   
4            0.086802  0.017040    0.513739   5.918536               1.0   

   leverage  conviction  zhangs_metric   jaccard  certainty  kulczynski  
0  0.019633    2.017035       0.943142  0.310

In [32]:
min_support = 0.005
min_confidence = 0.5
min_lift = 3

#running fp_growth
frequent_itemsets = fpgrowth(basket, min_support, use_colnames=True)

#generating rules
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_confidence)
rules = rules[rules['lift'] >= min_lift]

print(f"Number of rules: {len(rules)}")
print(rules.head())

Number of rules: 1796
                  antecedents         consequents  antecedent support  \
0          frozenset({22064})  frozenset({21232})            0.013091   
1          frozenset({22195})  frozenset({22196})            0.014361   
2          frozenset({21755})  frozenset({21754})            0.041012   
3  frozenset({21755, 85123A})  frozenset({21754})            0.014969   
4          frozenset({21756})  frozenset({21754})            0.014333   

   consequent support   support  confidence       lift  representativity  \
0            0.051341  0.007622    0.582278  11.341432               1.0   
1            0.015493  0.007484    0.521154  33.637183               1.0   
2            0.050568  0.021707    0.529293  10.467050               1.0   
3            0.050568  0.007540    0.503690   9.960739               1.0   
4            0.050568  0.008589    0.599229  11.850078               1.0   

   leverage  conviction  zhangs_metric   jaccard  certainty  kulczynski  
0  0.006

# Output Explaination

antecedents: {22064}  →  consequents: {21232}
Means: customers who bought product 22064 also bought product 21232

**The numbers**:

- antecedent support = 0.016707 → Product 22064 appears in 1.67% of all transactions
- consequent support = 0.069484 → Product 21232 appears in 6.94% of all transactions
- support = 0.010306 → Both products appear together in 1.03% of all transactions
- confidence = 0.616822 → 61.7% of customers who bought 22064 also bought 21232
- lift = 8.877 → Customers who bought 22064 are 8.8x more likely to buy 21232 than a random customer. This is a very strong association

**So in Summary**:

If a customer buys product 22064, there's a 61.7% chance they'll also buy product 21232, and this is 8.8 times stronger than random chance

# TF-IDF Vectors for Products

In this step, the textual descriptions of the products are converted into numerical vectors using the TF-IDF (Term Frequency-Inverse Document Frequency) technique. This allows the system to calculate the similarity between products based on their content rather than just transaction patterns. The cosine_similarity matrix is computed, which provides a score between 0 and 1 indicating how similar any two products are based on their descriptions.

In [15]:
tfidf = TfidfVectorizer(stop_words='english')
vectors = tfidf.fit_transform(metadata_df['Description'])
content_sim_matrix = cosine_similarity(vectors, vectors)
item_index_mapping = pd.Series(metadata_df.index, index=metadata_df['StockCode']).to_dict() #maps stockcode -> rowindex so we can find directly
reverse_index_mapping = {v: k for k, v in item_index_mapping.items()} #rowindex -> stockcode

In [16]:
def get_candidates(customer_id, df_transactions, ar_rules, similarity_matrix, top_k=5):
    
    #get the user's history
    user_history = set(df_transactions[df_transactions['Customer ID'] == customer_id]['StockCode'].unique())
    
    if not user_history:
        return {}, {}

    #generate AR Candidates
    ar_candidates = {}
    
    #filter rules where the antecedent is a subset of the user's history
    matching_rules = ar_rules[ar_rules['antecedents'].apply(lambda x: x.issubset(user_history))]
    
    for _, row in matching_rules.iterrows():
        for item in row['consequents']:
            if item not in user_history:
                
                ar_candidates[item] = max(ar_candidates.get(item, 0), row['confidence']) #store the highest confidence score for each candidate

    #generate Content-Based Candidates
    content_candidates = {}
    
    for item in user_history:
        if item in item_index_mapping:
            
            idx = item_index_mapping[item] 
            sim_scores = list(enumerate(similarity_matrix[idx])) #get similarity scores for this item against all others
            sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_k+1] #sort and take top k similar items
            
            for sim_idx, score in sim_scores:
                
                candidate_item = reverse_index_mapping.get(sim_idx)
                
                if candidate_item and candidate_item not in user_history:
                    content_candidates[candidate_item] = max(content_candidates.get(candidate_item, 0), score)
                    
    return ar_candidates, content_candidates

#test candidate generation for a sample user
ar_cands, ct_cands = get_candidates(13085, interaction_df, rules, content_sim_matrix)
print(f"Generated {len(ar_cands)} AR candidates and {len(ct_cands)} Content candidates.")

Generated 4 AR candidates and 185 Content candidates.


In [17]:
def rank_hybrid_recommendations(ar_candidates, content_candidates, alpha=0.5, k=5):
    
    combined_items = set(ar_candidates.keys()).union(set(content_candidates.keys()))
    final_scores = []

    for item in combined_items:
        ar_score = ar_candidates.get(item, 0)
        content_score = content_candidates.get(item, 0)
        score = (alpha * ar_score) + ((1 - alpha) * content_score) #weighted hybrid formula
        final_scores.append({
            'StockCode': item,
            'HybridScore': score,
            'AR_Score': ar_score,
            'Content_Score': content_score
        })

    
    ranked_list = sorted(final_scores, key=lambda x: x['HybridScore'], reverse=True) #sort by hybrid_score descending
    return pd.DataFrame(ranked_list).head(k)

#generate final recommendations for the test user
recommendations = rank_hybrid_recommendations(ar_cands, ct_cands, alpha=0.5, k=5)

#attach descriptions for readability
recommendations = recommendations.merge(
    metadata_df[['StockCode', 'Description']], on='StockCode', how='left'
)

print("Top 5 Hybrid Recommendations:")
recommendations[['StockCode', 'Description', 'HybridScore', 'AR_Score', 'Content_Score']]

Top 5 Hybrid Recommendations:


,StockCode,Description,HybridScore,AR_Score,Content_Score
0,22747,POPPY'S PLAYHOUSE BATHROOM,0.739144,0.733711,0.744577
1,22196,SMALL HEART MEASURING SPOONS,0.690245,0.521154,0.859336
2,22191,EAU DE NIL DINER WALL CLOCK,0.583516,0.526419,0.640614
3,22273,FELTCRAFT DOLL MOLLY,0.515764,0.515284,0.516245
4,22703,PINK CAT BOWL,0.461232,0.000000,0.922464


# Output Explanation
**Recommendation**:
If a user has interacted with items similar to "PACK OF 72 RETRO SPOT CAKE CASES," the system recommends the "PINK CAT BOWL."

**The Numbers**:
AR_Score = 0.000000: This indicates there is no direct transaction-based rule linking the user's past purchases to the "PINK CAT BOWL."

Content_Score = 0.925157: This shows that the product description of the "PINK CAT BOWL" is 92.5% similar to the user's past purchase history (calculated via TF-IDF cosine similarity).

HybridScore = 0.462579: This is the weighted average (50/50 split) of the AR and Content scores.

**So in Summary**:
Even though the transaction logs (AR) don't suggest this user would buy the "PINK CAT BOWL," the system recommends it because its product attributes strongly match the user's previous preferences. This confirms that the hybrid approach successfully uses content similarity to overcome the "cold start" limitation of association rules.

# Evaluation

## Precision, Recall, and F1

Think of it like a store employee suggesting products to a customer:
Precision@5 — "Of the 5 items you recommended, how many did the customer actually buy?"

Precision = |Recommended ∩ Actually Bought| / |Recommended|

Your result: 4.3% → Out of 5 recommendations, the customer bought ~0.21 items on average.
- High precision = our recommendations are relevant and trustworthy.
- Low precision = we're showing a lot of stuff nobody wants.

Recall@5 — "Of all the items the customer actually bought, how many did you recommend?"

Recall = |Recommended ∩ Actually Bought| / |Actually Bought|

Your result: 2.58% → You caught ~2.6% of what they ended up buying.

- High recall = we didn't miss much they might want.
- Low recall = we missed most of their interests.
--------------------
|                   | High Precision                       | High Recall                        |
| ----------------- | ------------------------------------ | ---------------------------------- |
| **What it means** | "When I suggest something, trust me" | "I won't let you miss anything"    |
| **Risk**          | Boring, safe recommendations         | Spammy, irrelevant recommendations |
| **Analogy**       | Specialist doctor                    | General practitioner               |

### F1 Score — The Balanced Metric
Since we can't maximize both at once, F1 is the harmonic mean — it punishes being terrible at either one:

F1 = 2 × (Precision × Recall) / (Precision + Recall)

Only high when both precision and recall are decent.
If precision = 100% but recall = 0%, F1 = 0. Useless.

That's why we going to add F1 as meric for better evaluation

In [23]:
def evaluate(alpha=0.5, top_k=5, test_ratio=0.2, n_customers=200):
    
    precisions = []
    recalls = []
    
    customer_items = interaction_df.groupby('Customer ID')['StockCode'].apply(list)
    customer_items = customer_items[customer_items.apply(len) >= 5]
    customer_items = customer_items.sample(n=n_customers, random_state=42)
    
    for customer_id, items in customer_items.items():
        
        split = int(len(items) * (1 - test_ratio))
        seed_items = items[:split]
        ground_truth = set(items[split:])
        seed_df = interaction_df[ #temporary df with only seed items for this customer
            (interaction_df['Customer ID'] == customer_id) & 
            (interaction_df['StockCode'].isin(seed_items))
        ]
        ar_cands, ct_cands = get_candidates(customer_id, seed_df, rules, content_sim_matrix, top_k=top_k) #get candidates for this specific customer
        recs = rank_hybrid_recommendations(ar_cands, ct_cands, alpha=alpha, k=top_k)
        recommended = set(recs['StockCode'].tolist())
        
        if len(recommended) == 0:
            continue
        
        #calculate once per customer, not per seed
        hits = recommended & ground_truth
        precisions.append(len(hits) / len(recommended))
        recalls.append(len(hits) / len(ground_truth))
        
    precision = sum(precisions)/len(precisions)
    recall = sum(recalls)/len(recalls)
    f1 = 2 * (precision*recall) / (precision+recall)
    
    print(f"Alpha={alpha} | Precision={precision:.4f} | Recall={recall:.4f} |  F1={f1:.4f}")

In [ ]:
for a in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    evaluate(alpha=a, top_k=5, n_customers=1000)

Alpha=0.0 | Precision=0.0283 | Recall=0.0248 |  F1=0.0264
